In [1]:
#loading from local
import pandas as pd

df = pd.read_parquet("/home/dilmurod/job/tts_project/data/karakalpak-audio-dataset/train-00000-of-00003.parquet")


In [2]:
# exploring dataset
df.head()

,audio,sentence_latin,sentence_cyrillic
0,{'bytes': b'RIFF\x8a!\x02\x00WAVEfmt \x10\x00\...,Internatta,Интернатта
1,{'bytes': b'RIFF\xa2\xfe\x03\x00WAVEfmt \x10\x...,Oktyabr ayınıń sońǵı kúnleri.,Октябрь айының соңғы күнлери.
2,{'bytes': b'RIFF\x88[\t\x00WAVEfmt \x10\x00\x0...,"Kún uyasına sıymay dolanıp, qıp-qızıl shoqtay ...","Күн уясына сыймай доланып, қып-қызыл шоқтай жа..."
3,{'bytes': b'RIFF(\xcf\x06\x00WAVEfmt \x10\x00\...,“Kese etek”ten Qońıratqa qaray sozılǵan úlken ...,“Кесе етек”тен Қоңыратқа қарай созылған үлкен ...
4,{'bytes': b'RIFF\x98\xb3\x02\x00WAVEfmt \x10\x...,Jeter jerde el joq.,Жетер жерде ел жоқ.


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 994 entries, 0 to 993
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   audio              994 non-null    object
 1   sentence_latin     994 non-null    str   
 2   sentence_cyrillic  994 non-null    str   
dtypes: object(1), str(2)
memory usage: 201.6+ KB


In [4]:
# checking for n/a values
df.isna().sum()

audio                0
sentence_latin       0
sentence_cyrillic    0
dtype: int64

In [5]:
# how the auido and other data stored
df.iloc[0]

audio                {'bytes': b'RIFF\x8a!\x02\x00WAVEfmt \x10\x00\...
sentence_latin                                              Internatta
sentence_cyrillic                                           Интернатта
Name: 0, dtype: object

In [6]:
# let's learn audio structure
audio = df["audio"].iloc[0]

print(type(audio))
print(audio.keys())
print(len(audio["bytes"]))
print(audio["bytes"])
print(audio['path'])
print(type(audio["bytes"]))
print(type(audio["path"]))

<class 'dict'>
dict_keys(['bytes', 'path'])
139666
b'RIFF\x8a!\x02\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00D\xac\x00\x00\x88X\x01\x00\x02\x00\x10\x00LIST\x1a\x00\x00\x00INFOISFT\r\x00\x00\x00Lavf61.7.100\x00\x00dataD!\x02\x00\x06\x00\x0f\x00\t\x00\x05\x00\xfe\xff\x08\x00\t\x00\x07\x00\x04\x00\x04\x00\x14\x00\x0c\x00\xfb\xff\xff\xff\r\x00\x0c\x00\x07\x00\x16\x00\x0e\x00\xf8\xff\xfa\xff\x07\x00\x03\x00\x05\x00\x02\x00\x07\x00\x06\x00\x07\x00\r\x00\x08\x00\n\x00\x0e\x00\x1b\x00!\x00\x16\x00\x08\x00\t\x00\x0e\x00\x01\x00\r\x00\x0e\x00\x02\x00\x0c\x00\n\x00\xfe\xff\xff\xff\xff\xff\xf3\xff\xf8\xff\x03\x00\xfc\xff\xf6\xff\x00\x00\xfd\xff\xf3\xff\xe5\xff\xeb\xff\xf1\xff\x03\x00\x00\x00\x00\x00\xfb\xff\xf3\xff\xeb\xff\xf2\xff\t\x00\x03\x00\xf4\xff\xfd\xff\x04\x00\xf5\xff\xec\xff\xf0\xff\xec\xff\xe7\xff\xf7\xff\xf6\xff\xf5\xff\xf5\xff\x01\x00\xf7\xff\xf1\xff\xf3\xff\xff\xff\xfa\xff\xf7\xff\xfd\xff\x04\x00\xf9\xff\xef\xff\xe8\xff\xeb\xff\xf2\xff\xf1\xff\xeb\xff\xe0\xff\xe8\xff\xfb\xff\xee\xff\

In [10]:
# lets listen one sample

import subprocess
from pathlib import Path

i = 0

# Get audio + text
text = df.iloc[i]["sentence_latin"]
audio_bytes = df.iloc[i]["audio"]["bytes"]

# Save audio
path = Path(f"sample_{i}.wav")
path.write_bytes(audio_bytes)

print("Text:", text)
print("Opening:", path.absolute())

# Open with the system's default media player
subprocess.Popen(["xdg-open", str(path.absolute())])

Text: Internatta
Opening: /home/dilmurod/job/tts_project/notebooks/sample_0.wav


<Popen: returncode: None args: ['xdg-open', '/home/dilmurod/job/tts_project/...>

In [11]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print()

print("Missing values:")
print(df.isna().sum())

Rows: 994
Columns: ['audio', 'sentence_latin', 'sentence_cyrillic']

Missing values:
audio                0
sentence_latin       0
sentence_cyrillic    0
dtype: int64


In [12]:
audio = df["audio"].iloc[0]

print("Audio type:", type(audio))
print("Audio keys:", audio.keys())
print("Path:", audio["path"])
print("Audio bytes:", len(audio["bytes"]))

Audio type: <class 'dict'>
Audio keys: dict_keys(['bytes', 'path'])
Path: kaa_audio_1000001.wav
Audio bytes: 139666


In [13]:
from pathlib import Path

extensions = df["audio"].apply(
    lambda x: Path(x["path"]).suffix.lower()
)

print(extensions.value_counts())

audio
.wav    994
Name: count, dtype: int64


In [14]:
# duration, sample rate (Hz), total hours

# sr is sample rate -> so duration is len / sr
# total_hours= total_seconds / 3600

import io
import soundfile as sf


sample_rates = []
durations = []

for audio in df["audio"]:
    data, sr = sf.read(io.BytesIO(audio["bytes"]))

    sample_rates.append(sr)
    durations.append(len(data) / sr)

print("Sample rates:", sample_rates[:10])
print("Durations:", durations[:10])
print(len(df), 'sample in df')

Sample rates: [44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100]
Durations: [1.5826303854875283, 2.967392290249433, 6.952222222222222, 5.0587074829931975, 2.0065532879818595, 9.693514739229025, 3.6174149659863946, 3.92827664399093, 6.302222222222222, 9.693514739229025]
994 sample in df


In [15]:
total_hours=sum(durations)/3600

print("Total hours of df is",total_hours)

Total hours of df is 1.1957327412446461
